In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


sns.set(style="whitegrid", font="Malgun Gothic")  # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False        # 마이너스 깨짐 방지

# 데이터 경로
BASE_DIR = Path("C:/dev/SKN18-FINAL-1TEAM/data/actual_transaction_price")

train_path = BASE_DIR / "월세_train(24.08~25.08).csv"
test_path  = BASE_DIR / "월세_test(25.09~25.10).csv"

train = pd.read_csv(train_path, encoding="utf-8-sig")
test  = pd.read_csv(test_path,  encoding="utf-8-sig")

print("train:", train.shape)
print("test :", test.shape)
train.head()

train: (423570, 17)
test : (52236, 17)


,자치구명,법정동명,층,연월,임대면적,보증금(만원),임대료(만원),건축년도,건물용도,소비자물가,무담보콜금리,KORIBOR,CD,기업대출,전세자금대출,변동형주택담보대출,기준금리
0,마포구,노고산동,5.0,2024-12,31.12,8000,50,1998.0,오피스텔,1.9,3.055,3.31,3.35,4.62,4.34,4.32,3.00
1,종로구,명륜3가,2.0,2024-09,20.96,500,76,1996.0,연립다세대,1.6,3.529,3.49,3.52,4.77,4.05,4.08,3.50
2,강북구,수유동,3.0,2024-09,20.00,1000,70,2023.0,연립다세대,1.6,3.529,3.49,3.52,4.77,4.05,4.08,3.50
3,용산구,용산동2가,2.0,2024-11,21.84,500,60,2020.0,연립다세대,1.5,3.245,3.40,3.42,4.76,4.43,4.25,3.25
4,종로구,명륜3가,4.0,2024-08,47.58,500,40,1997.0,연립다세대,2.0,3.531,3.45,3.50,4.67,3.82,4.04,3.50


In [2]:
import numpy as np
import pandas as pd

def add_deposit_per_pyeong(
    df: pd.DataFrame,
    deposit_col: str = "보증금",
    monthly_rent_col: str = "월세",
    area_m2_col: str = "전용면적_m2",
    rent_multiplier: float = 100.0,  # 월세*100을 보증금으로 환산하는 계수
    out_col: str = "평당환산보증금"
) -> pd.DataFrame:
    df = df.copy()

    # 환산보증금 = 보증금 + 월세 * 계수
    df["환산보증금"] = df[deposit_col] + df[monthly_rent_col] * rent_multiplier

    # 평수 = m2 / 3.3
    df["전용면적_평"] = df[area_m2_col] / 3.3

    # 평당 환산보증금
    df[out_col] = df["환산보증금"] / df["전용면적_평"]

    # 0 또는 음수/NaN 제거
    df.loc[df["전용면적_평"] <= 0, out_col] = np.nan

    return df


In [13]:
def add_quantile_bins_by_dong_usage(
    df: pd.DataFrame,
    value_col: str = "평당환산보증금",
    quantiles=(0.33, 0.67),
    labels=("저", "중", "고"),
    min_group_size: int = 10,
    out_col: str = "보증금_구간"
) -> pd.DataFrame:
    df = df.copy()

    def _bin_group(g: pd.DataFrame) -> pd.Series:
        vals = g[value_col].dropna()

        # 샘플이 너무 적으면 전체를 '중'으로
        if len(vals) < min_group_size:
            return pd.Series(["중"] * len(g), index=g.index)

        qs = vals.quantile(quantiles)
        q1, q2 = qs.iloc[0], qs.iloc[1]

        # 분위수 계산이 비정상(무한/NaN)이거나 q1==q2 이면 전체를 '중'으로
        if (not np.isfinite(q1)) or (not np.isfinite(q2)) or np.isclose(q1, q2):
            return pd.Series(["중"] * len(g), index=g.index)

        bins = [-np.inf, q1, q2, np.inf]
        return pd.cut(g[value_col], bins=bins, labels=labels)

    df[out_col] = (
        df.groupby(["법정동명", "건물용도"], group_keys=False)
          .apply(_bin_group)
    )
    return df


In [14]:
# 1) 평당 환산보증금 계산
df1 = add_deposit_per_pyeong(
    train,
    deposit_col="보증금(만원)",
    monthly_rent_col="임대료(만원)",
    area_m2_col="임대면적"
)

# 2) 행정동×건물용도별 분위수 구간(저/중/고) 부여
df2 = add_quantile_bins_by_dong_usage(
    df1,
    value_col="평당환산보증금",
    quantiles=(0.33, 0.67),
    labels=("저", "중", "고"),
    out_col="보증금_구간"
)

# 3) 분위수 구간이 논리적으로 맞는지 점검
check_bins_logic(
    df2,
    value_col="평당환산보증금",
    bin_col="보증금_구간"
)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_28144\3768489652.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_bin_group)



[전체 구간별 요약 통계]
    법정동명   건물용도 보증금_구간  count         mean          min          max
0    가락동    아파트      고   1183  3111.914222  2329.959991  5570.332481
1    가락동    아파트      저   1196  1219.614269    78.180526  1452.506394
2    가락동    아파트      중   1232  1856.462896  1452.643772  2329.685845
3    가락동  연립다세대      고    603  2334.269268  1653.566516  4157.315266
4    가락동  연립다세대      저    603   840.458558    59.393604  1102.695478
5    가락동  연립다세대      중    620  1344.933029  1103.210401  1652.034525
6    가락동   오피스텔      고     37  2807.187834  1347.542243  4560.901691
7    가락동   오피스텔      저     47   813.974983   429.280984   919.220056
8    가락동   오피스텔      중     56  1127.134163   943.755958  1305.538649
9   가리봉동    아파트      고     13  1295.382237  1211.211211  1401.604530
10  가리봉동    아파트      저     14   980.775447   748.916856  1079.676674
11  가리봉동    아파트      중     12  1154.152079  1122.448980  1206.937235
12  가리봉동  연립다세대      고     77  2266.346776  1950.629723  3066.525871
13  가리봉동  연립다세대   

In [15]:
import numpy as np
import pandas as pd

# 분류에 썼던 것과 동일한 기준
min_group_size = 10
q_low, q_high = 0.33, 0.67

# 1) 행정동×건물용도별 통계 + fallback 여부 계산
group_stats = (
    df1.dropna(subset=["평당환산보증금"])
       .groupby(["법정동명", "건물용도"])["평당환산보증금"]
       .agg(
           count="count",
           q33=lambda x: x.quantile(q_low),
           q67=lambda x: x.quantile(q_high),
       )
       .reset_index()
)

group_stats["use_fallback"] = (
    (group_stats["count"] < min_group_size) |
    (np.isclose(group_stats["q33"], group_stats["q67"]))
)

# 2) 그룹 기준 비율
total_groups = len(group_stats)
fallback_groups = int(group_stats["use_fallback"].sum())
print(f"전체 그룹 수: {total_groups}")
print(f"fallback 그룹 수: {fallback_groups} ({fallback_groups/total_groups:.2%})")

# 3) 샘플(매물) 기준 비율
df_tmp = df1.merge(
    group_stats[["법정동명", "건물용도", "use_fallback"]],
    on=["법정동명", "건물용도"],
    how="left",
)

total_samples = len(df_tmp)
fallback_samples = int(df_tmp["use_fallback"].sum())
print(f"\n전체 매물 수: {total_samples}")
print(f"fallback 매물 수: {fallback_samples} ({fallback_samples/total_samples:.2%})")

# 4) 어떤 그룹들이 fallback인지 일부 확인
print("\n[fallback 그룹 예시 상위 20개]")
print(
    group_stats[group_stats["use_fallback"]]
    .sort_values("count")
    .head(20)
)


전체 그룹 수: 921
fallback 그룹 수: 100 (10.86%)

전체 매물 수: 423570
fallback 매물 수: 1626 (0.38%)

[fallback 그룹 예시 상위 20개]
      법정동명   건물용도  count          q33          q67  use_fallback
34      계동  연립다세대      1   314.285714   314.285714          True
228    동숭동    아파트      1  1028.816987  1028.816987          True
153  당산동5가  연립다세대      1  1493.982808  1493.982808          True
193    도원동  연립다세대      1  4020.360481  4020.360481          True
271   명륜3가    아파트      1   990.099010   990.099010          True
529    신촌동   오피스텔      1   915.077990   915.077990          True
834   필동3가    아파트      1   453.608247   453.608247          True
100  남산동2가    아파트      2  1106.321839  1106.321839          True
213  동선동4가    아파트      2  1103.037349  1103.037349          True
128    누상동    아파트      2  1663.306452  1663.306452          True
513    신영동    아파트      2   964.639321   964.639321          True
501  신문로2가    아파트      2  3056.867847  3056.867847          True
391    삼청동  연립다세대      2   923.782296  1022.